[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eygpcr/biyofizik2026-martini/blob/main/notebooks/04_martini_input.ipynb)

# Oturum 4 — Martini 3 Girdi Hazırlama

**Biyofizik 2026 Kursu · 25 Ağustos 2026 · Dr. Öğr. Üyesi Ekrem Yaşar**

Sabah oturumlarında CHARMM-GUI arayüzü ile atomistik olarak hazırlanan protein
(`6JOD` A zinciri, anjiyotensin II tip-2 reseptörü), bu not defterinde komut
satırı araçlarıyla ve Martini 3 kaba-taneli modeli kullanılarak yeniden
hazırlanmaktadır.

Kursta üretim simülasyonu koşulmamaktadır; amaç simülasyona girecek sistemin
kurulmasıdır.

**İşlem sırası**

1. Google Drive bağlanması ve çalışma klasörünün oluşturulması
2. Yazılım kurulumu
3. Yapının hazırlanması (A zincirinin ayıklanması)
4. `martinize2` ile kaba-taneli modele dönüştürme
5. Martini 3 kuvvet alanı dosyalarının indirilmesi
6. `insane` ile membran, çözücü ve iyon eklenmesi
7. Topoloji dosyasının düzeltilmesi
8. `gmx grompp` ile doğrulama
9. Sistemin görselleştirilmesi
10. Çıktıların Drive'a kaydedilmesi


---
## 1. Google Drive bağlanması

Colab çalışma zamanı sonlandığında üretilen dosyalar silinmektedir. Bu nedenle
çıktılar Google Drive üzerinde kalıcı bir klasöre kaydedilecektir.

Aşağıdaki hücre çalıştırıldığında Google hesabına erişim izni istenecektir;
açılan pencerede kendi hesabınızı seçip izin veriniz.

Oluşturulacak klasör yapısı:

```
Drive'ım/
└── Biyofizik2026_Martini/
    └── martini_input/          <- bu oturum
        ├── girdi/              ham ve hazırlanmış yapılar
        ├── cikti/              üretilen koordinat, topoloji ve parametre dosyaları
        └── gorseller/          çizimler
```

**Not.** Oturum 6 aynı `Biyofizik2026_Martini` klasörü altında ayrı bir
`bentopy/` alt klasörü kullanmaktadır; oturumların çıktıları karışmamaktadır.


In [ ]:
from google.colab import drive
import os, shutil

drive.mount('/content/drive')

# --- Kalici klasor yapisi (Google Drive) ---
DRIVE_KOK  = '/content/drive/MyDrive/Biyofizik2026_Martini'
OTURUM     = os.path.join(DRIVE_KOK, 'martini_input')
D_GIRDI    = os.path.join(OTURUM, 'girdi')
D_CIKTI    = os.path.join(OTURUM, 'cikti')
D_GORSEL   = os.path.join(OTURUM, 'gorseller')

for d in (DRIVE_KOK, OTURUM, D_GIRDI, D_CIKTI, D_GORSEL):
    os.makedirs(d, exist_ok=True)

# --- Hizli yerel calisma dizini ---
# Agir dosya islemleri Drive uzerinde yavas oldugundan yerelde calisilir,
# uretilen dosyalar adim adim Drive'a kopyalanir.
CALISMA = '/content/calisma'
os.makedirs(CALISMA, exist_ok=True)
os.chdir(CALISMA)

print('Drive klasoru :', OTURUM)
print('Calisma dizini:', os.getcwd())


### Kaydetme yardımcı fonksiyonu

Her adımın sonunda üretilen dosyalar bu fonksiyonla Drive'a kopyalanacaktır.


In [ ]:
import glob

def kaydet(desenler, hedef, sessiz=False):
    """Verilen dosya desenlerini Drive'daki hedef klasore kopyalar."""
    kopyalanan = []
    for desen in desenler:
        for dosya in glob.glob(desen):
            if os.path.isfile(dosya):
                shutil.copy(dosya, hedef)
                kopyalanan.append(os.path.basename(dosya))
    if not sessiz:
        if kopyalanan:
            print(f'Drive\'a kaydedildi ({os.path.basename(hedef)}/):')
            for k in sorted(kopyalanan):
                print('  -', k)
        else:
            print('Kopyalanacak dosya bulunamadi:', desenler)
    return kopyalanan


---
## 2. Yazılım kurulumu

Aşağıdaki hücrenin çalışma süresi yaklaşık 3–5 dakikadır ve oturum başına bir
kez çalıştırılması yeterlidir.

| Paket | İşlevi |
|---|---|
| `vermouth` | `martinize2` komutunu sağlamaktadır |
| `insane` | Membran ve kutu inşası |
| `gromacs` | `gmx grompp` ile doğrulama |
| `dssp` | İkincil yapı tayini (`mkdssp`) |


In [ ]:
%%capture
!pip install -q vermouth insane
!apt-get -qq update
!apt-get -qq install -y gromacs dssp


In [ ]:
# Kurulumun dogrulanmasi
!martinize2 --version 2>&1 | head -2
!insane --help 2>&1 | head -3
!gmx --version 2>&1 | grep -i 'GROMACS version'
!which mkdssp || echo 'mkdssp bulunamadi; asagida -ss secenegi kullanilacaktir'


---
## 3. Yapının hazırlanması

`6JOD` yapısı indirilerek yalnızca A zinciri (AT2R) ayıklanmaktadır.

Yapıda yer alan C zinciri BRIL füzyonunu, H ve L zincirleri ise Fab fragmanını
temsil etmektedir. Bu bileşenler kristalizasyon amacıyla eklenmiş deneysel
araçlar olup fizyolojik ortamda bulunmamaktadır; sisteme dâhil edilmemektedir.

Bu oturumda ligant (B zinciri) da alınmamakta, yalnızca protein ve membrandan
oluşan sade bir sistem kurulmaktadır.


In [ ]:
!wget -q https://files.rcsb.org/download/6JOD.pdb -O 6jod.pdb

# Yalnizca A zincirinin ATOM kayitlari (su ve hetero gruplar haric)
kept = []
for line in open('6jod.pdb'):
    if line.startswith('ATOM  ') and line[21] == 'A':
        kept.append(line)
kept.append('END\n')
open('at2r.pdb','w').writelines(kept)

resids = sorted({int(l[22:26]) for l in kept if l.startswith('ATOM')})
print(f'A zinciri: {len(resids)} rezidu ({resids[0]}-{resids[-1]}), {len(kept)-1} atom')

kaydet(['6jod.pdb', 'at2r.pdb'], D_GIRDI)


**Beklenen sonuç.** 35–340 aralığında 306 rezidü.
Dizide 312 rezidü bulunmakta olup 341–346 aralığındaki C-terminal uzantı
çözülmemiştir. Zincir içi kopukluk bulunmadığından ilmik modellemesine gerek
duyulmamaktadır.


---
## 4. `martinize2` ile kaba-taneli modele dönüştürme

Atomistik protein Martini 3 etkileşim merkezlerine dönüştürülmektedir.

| Parametre | İşlevi | Önemi |
|---|---|---|
| `-ff martini3001` | Martini 3 kuvvet alanı | Martini 2 ile karıştırılmamalıdır |
| `-dssp` | İkincil yapının belirlenmesi | Merkez tipleri ikincil yapıya bağlıdır |
| `-elastic` | Elastik ağ tanımlanması | Uygulanmadığında yapısal bütünlük korunamamaktadır |
| `-ef 700` | Yay kuvvet sabiti (kJ mol⁻¹ nm⁻²) | Yüksek değer aşırı rijitlik, düşük değer yapısal bozulma |
| `-el 0.5 -eu 0.9` | Yay mesafe aralığı (nm) | Bağlanacak merkez çiftlerini belirlemektedir |


In [ ]:
!martinize2 \
  -f at2r.pdb \
  -o topol.top \
  -x at2r_cg.pdb \
  -ff martini3001 \
  -dssp \
  -elastic -ef 700 -el 0.5 -eu 0.9 -ea 0 -ep 0 \
  -maxwarn 10


### `-dssp` seçeneğinin çalışmaması hâlinde

`mkdssp` bulunamadığında ikincil yapı doğrudan tanımlanabilmektedir. Aşağıdaki
hücre yalnızca önceki hücrenin hata vermesi durumunda çalıştırılmalıdır; tüm
diziyi heliks olarak atamakta olup bir GPCR için yaklaşık ancak kabul edilebilir
bir çözümdür.

Yayımlanacak çalışmalarda `mkdssp` kurulumunun yapılması önerilmektedir.


In [ ]:
# Yalnizca onceki hucre -dssp nedeniyle hata verdiginde calistirilmalidir
# !martinize2 -f at2r.pdb -o topol.top -x at2r_cg.pdb -ff martini3001 \
#   -ss $(python3 -c "print('H'*306)") \
#   -elastic -ef 700 -el 0.5 -eu 0.9 -ea 0 -ep 0 -maxwarn 10


In [ ]:
# Indirgeme oraninin hesaplanmasi
aa = sum(1 for l in open('at2r.pdb') if l.startswith('ATOM'))
cg = sum(1 for l in open('at2r_cg.pdb') if l.startswith(('ATOM','HETATM')))
print(f'Atomistik model  : {aa:>6} atom')
print(f'Kaba-taneli model: {cg:>6} etkilesim merkezi')
print(f'Indirgeme orani  : {aa/cg:.1f}')
print()

kaydet(['at2r_cg.pdb', 'molecule_*.itp', 'topol.top'], D_CIKTI)


**Elastik ağın gerekliliği.** Martini kuvvet alanı, etkileşim merkezleri
arasındaki potansiyeller aracılığıyla proteinin üçüncül yapısını koruyamamaktadır.
Yapı, harmonik yaylardan oluşan bir ağ ile kısıtlanmaktadır.

Bu yaklaşımın sonucu olarak model protein katlanma, açılma ve büyük ölçekli
konformasyonel değişim gösterememektedir. Dolayısıyla Martini modeli ile bir
GPCR'ın aktivasyon geçişi incelenememektedir. İncelenebilecek süreçler lipit
etkileşimleri, oligomerizasyon ve difüzyondur.


In [ ]:
# Uretilen topolojinin incelenmesi
import glob
itp = sorted(glob.glob('molecule_*.itp'))[0]
lines = open(itp).read().splitlines()
print('Dosya:', itp, '|', len(lines), 'satir\n')
for i, l in enumerate(lines):
    if l.strip().startswith('['):
        print(f'{i:>6}  {l.strip()}')


---
## 5. Martini 3 kuvvet alanı dosyalarının indirilmesi

`martinize2` proteinin topolojisini üretmiştir. Ayrıca Martini kuvvet alanının
genel parametre dosyaları (etkileşim merkezi tanımları, lipitler, çözücü ve
iyonlar) gerekmektedir.

Kaynak: [marrink-lab/martini-forcefields](https://github.com/marrink-lab/martini-forcefields)

Ana parametre dosyasının boyutu yaklaşık 16 MB olup indirme süresi bir dakikayı
bulabilmektedir. Bu dosyalar her oturumda yeniden indirilebildiğinden Drive'a
kaydedilmemektedir.


In [ ]:
BASE = 'https://raw.githubusercontent.com/marrink-lab/martini-forcefields/main/martini_forcefields/regular/v3.0.0/gmx_files'
files = [
    'martini_v3.0.0.itp',
    'martini_v3.0.0_solvents_v1.itp',
    'martini_v3.0.0_ions_v1.itp',
    'martini_v3.0.0_phospholipids_v1.itp',
]
for f in files:
    !wget -q {BASE}/{f} -O {f}
!ls -lh martini_v3.0.0*.itp


---
## 6. `insane` ile membran, çözücü ve iyon eklenmesi

| Parametre | İşlevi |
|---|---|
| `-box 12,12,14` | Kutu boyutları (nm) |
| `-l POPC:1` | Lipit bileşimi |
| `-sol W` | Martini standart su modeli (bir merkez yaklaşık dört su molekülü) |
| `-salt 0.15` | 0.15 M NaCl |
| `-center` | Proteinin kutu merkezine yerleştirilmesi |

Çok bileşenli membran için: `-l POPC:7 -l POPE:2 -l CHOL:1`


In [ ]:
!insane \
  -f at2r_cg.pdb \
  -o sistem.gro \
  -p sistem_insane.top \
  -pbc square \
  -box 12,12,14 \
  -l POPC:1 \
  -sol W \
  -salt 0.15 \
  -center \
  -dm 0


In [ ]:
print('--- insane tarafindan uretilen topoloji ---')
print(open('sistem_insane.top').read())

n = int(open('sistem.gro').read().splitlines()[1])
print(f'Toplam parcacik sayisi: {n:,}')


**Değerlendirme.** Aynı hacimdeki atomistik bir sistem yaklaşık on kat daha
fazla parçacık içerecekti. Buna ek olarak Martini modeli daha büyük zaman adımı
kullanılmasına olanak vermektedir.


---
## 7. Topoloji dosyasının düzeltilmesi

Bu adım, iş akışında en sık hata alınan noktadır.

`insane` bir topoloji dosyası üretmekte, ancak `#include` yönergeleri eksik veya
hatalı olmaktadır; araç proteinin topolojisini tanımamaktadır. Dosya aşağıdaki
hücrede yeniden oluşturulmaktadır.

Yönerge sırası önemlidir: önce genel kuvvet alanı tanımları, ardından molekül
topolojileri yer almalıdır.


In [ ]:
import re, glob

prot_itp = sorted(glob.glob('molecule_*.itp'))[0]

# insane tarafindan uretilen [ molecules ] bolumu
raw = open('sistem_insane.top').read()
mols = raw.split('[ molecules ]')[1].strip().splitlines()
mols = [m for m in mols if m.strip() and not m.strip().startswith(';')]

# Protein molekul adinin martinize2 ciktisiyla eslestirilmesi
prot_name = None
for l in open(prot_itp):
    if l.strip().startswith('[ moleculetype ]'):
        continue
    if l.strip() and not l.strip().startswith((';','[')) and prot_name is None:
        prot_name = l.split()[0]
        break
print('Protein molekul adi:', prot_name)

fixed = []
for m in mols:
    parts = m.split()
    if parts[0].lower().startswith('protein'):
        fixed.append(f'{prot_name}   {parts[1]}')
    else:
        fixed.append(m.strip())

top = f'''; Biyofizik 2026 Kursu - Oturum 4
; AT2R (6JOD A zinciri) - Martini 3 - POPC membran

#include "martini_v3.0.0.itp"
#include "martini_v3.0.0_solvents_v1.itp"
#include "martini_v3.0.0_ions_v1.itp"
#include "martini_v3.0.0_phospholipids_v1.itp"
#include "{prot_itp}"

[ system ]
AT2R in POPC membrane (Martini 3)

[ molecules ]
''' + '\n'.join(fixed) + '\n'

open('sistem.top','w').write(top)
print('\n--- sistem.top ---')
print(top)

kaydet(['sistem.gro', 'sistem.top', 'sistem_insane.top'], D_CIKTI)


---
## 8. `gmx grompp` ile doğrulama

Bu adımda simülasyon yürütülmemektedir. Koordinat, topoloji ve parametre
dosyalarının birbiriyle tutarlılığı sınanmaktadır.

`.tpr` dosyasının üretilebilmesi, hazırlanan girdinin geçerli olduğunu
göstermektedir. Oturumun hedefi bu doğrulamanın sağlanmasıdır.


In [ ]:
mdp = '''; Martini 3 - enerji minimizasyonu
integrator               = steep
nsteps                   = 500
emtol                    = 100
emstep                   = 0.01

nstlist                  = 20
cutoff-scheme            = Verlet
verlet-buffer-tolerance  = 0.005

coulombtype              = reaction-field
rcoulomb                 = 1.1
epsilon_r                = 15
vdw_type                 = cutoff
vdw-modifier             = Potential-shift-verlet
rvdw                     = 1.1
'''
open('minimization.mdp','w').write(mdp)

!gmx grompp -f minimization.mdp -c sistem.gro -p sistem.top -o em.tpr -maxwarn 5


In [ ]:
import os
if os.path.exists('em.tpr'):
    print('em.tpr uretildi:', os.path.getsize('em.tpr'), 'bayt')
    print('Girdi hazirligi tamamlanmistir; bu dosya ile simulasyon yurutulebilir.')
else:
    print('em.tpr uretilemedi. Yukaridaki hata iletisi incelenmelidir.')
print()

kaydet(['minimization.mdp', 'em.tpr'], D_CIKTI)


Sabah oturumunda grafik arayüz ile gerçekleştirilen işlem, bu bölümde komut
satırı araçlarıyla ve kaba-taneli çözünürlükte tamamlanmıştır. Komut satırı
yaklaşımının belirleyici üstünlüğü, işlemin tekrarlanabilir ve raporlanabilir
olmasıdır.


---
## 9. Sistemin görselleştirilmesi

Kurulan sistemin doğru olup olmadığı, membran normali (z ekseni) boyunca
bileşenlerin dağılımına bakılarak denetlenebilir. Beklenen görünüm:

- Lipitler kutunun ortasında dar bir bant oluşturmalıdır (çift tabaka)
- Çözücü membranın iki yanında toplanmalı, membran içinde bulunmamalıdır
- Protein membranı kat etmeli, her iki yana taşmalıdır

Çizim `gorseller/` klasörüne kaydedilmektedir.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

LIPIT  = {'POPC','POPE','POPS','POPG','CHOL','DOPC','DPPC','DOPE'}
COZUCU = {'W','WF','PW'}
IYON   = {'NA','CL','NA+','CL-','ION','K'}

def gro_oku(path, max_atom=1_200_000):
    """GRO dosyasindan rezidu adlarini ve z koordinatlarini okur."""
    satirlar = open(path).read().splitlines()
    n = int(satirlar[1])
    atomlar = satirlar[2:2+n]
    if n > max_atom:
        atomlar = atomlar[::(n // max_atom + 1)]
    res = np.array([s[5:10].strip() for s in atomlar])
    z   = np.array([float(s[36:44]) for s in atomlar])
    return res, z

def profil_ciz(gro, png, baslik):
    res, z = gro_oku(gro)
    diger = list(LIPIT | COZUCU | IYON)
    gruplar = {
        'Protein': ~np.isin(res, diger),
        'Lipit'  : np.isin(res, list(LIPIT)),
        'Cozucu' : np.isin(res, list(COZUCU)),
        'Iyon'   : np.isin(res, list(IYON)),
    }
    kenar = np.linspace(z.min(), z.max(), 120)
    plt.figure(figsize=(9, 4.5))
    for ad, maske in gruplar.items():
        if maske.sum() == 0:
            continue
        plt.hist(z[maske], bins=kenar, histtype='step', lw=1.6,
                 label=f'{ad} (n={maske.sum():,})')
    plt.xlabel('z (nm)'); plt.ylabel('Parcacik sayisi')
    plt.title(baslik); plt.legend(); plt.grid(alpha=.3)
    plt.tight_layout()
    plt.savefig(png, dpi=150)
    plt.show()
    print('Kaydedildi:', png)

profil_ciz('sistem.gro',
           os.path.join(D_GORSEL, 'sistem_z_profili.png'),
           'AT2R / POPC sistemi - z ekseni boyunca dagilim')


In [ ]:
# Sistem bilesiminin ozeti
res, z = gro_oku('sistem.gro')
diger = list(LIPIT | COZUCU | IYON)
sayim = {
    'Protein': int((~np.isin(res, diger)).sum()),
    'Lipit'  : int(np.isin(res, list(LIPIT)).sum()),
    'Cozucu' : int(np.isin(res, list(COZUCU)).sum()),
    'Iyon'   : int(np.isin(res, list(IYON)).sum()),
}
toplam = sum(sayim.values())

plt.figure(figsize=(6.5, 4))
plt.bar(list(sayim), list(sayim.values()), color=['#4C72B0','#DD8452','#55A868','#C44E52'])
plt.ylabel('Parcacik sayisi'); plt.yscale('log')
plt.title('Sistem bilesimi')
for i, (k, v) in enumerate(sayim.items()):
    plt.text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
png = os.path.join(D_GORSEL, 'sistem_bilesimi.png')
plt.savefig(png, dpi=150); plt.show()

print('Kaydedildi:', png)
for k, v in sayim.items():
    print(f'  {k:<8}: {v:>8,}  (%{100*v/toplam:.1f})')


---
## 10. Çıktıların Drive'a kaydedilmesi

Önceki adımlarda dosyalar zaten kademeli olarak kaydedilmiştir. Aşağıdaki hücre
kalan dosyaları da kopyalayarak klasör içeriğini listelemektedir.


In [ ]:
kaydet(['at2r.pdb','at2r_cg.pdb','molecule_*.itp','topol.top'], D_CIKTI, sessiz=True)
kaydet(['sistem.gro','sistem.top','sistem_insane.top','minimization.mdp','em.tpr'], D_CIKTI, sessiz=True)

print('Google Drive icerigi:', OTURUM.replace('/content/drive/MyDrive', "Drive'im"))
print()
for alt in ('girdi', 'cikti', 'gorseller'):
    yol = os.path.join(OTURUM, alt)
    dosyalar = sorted(os.listdir(yol))
    print(f'{alt}/ ({len(dosyalar)} dosya)')
    for d in dosyalar:
        boyut = os.path.getsize(os.path.join(yol, d))
        print(f'    {d:<28} {boyut:>10,} bayt')
    print()


### İsteğe bağlı: bilgisayara indirme

Dosyalar Drive'da saklandığından bu adım gerekli değildir. Yerel bir kopya
isteyen katılımcılar aşağıdaki hücreyi çalıştırabilir.


In [ ]:
# import shutil
# from google.colab import files
# arsiv = shutil.make_archive('/content/oturum4_ciktilar', 'zip', OTURUM)
# files.download(arsiv)


---
## Sık karşılaşılan hata iletileri

| Hata iletisi | Nedeni | Çözümü |
|---|---|---|
| `Atomtype X not found` | Kuvvet alanı parametre dosyası eksik | 5. bölümdeki indirmeler denetlenmelidir |
| `number of coordinates does not match topology` | `[ molecules ]` bölümündeki sayılar hatalı | 7. bölüm yeniden çalıştırılmalıdır |
| `Unknown molecule type Protein` | Protein adı topoloji dosyalarında farklı | 7. bölüm bu düzeltmeyi otomatik yapmaktadır |
| `mkdssp not found` | DSSP kurulu değil | 4. bölümdeki `-ss` seçeneği kullanılmalıdır |
| `System has non-zero total charge` | Yuvarlama kaynaklı; olağandır | `-maxwarn` ile geçilebilir |
| LINCS uyarısı veya sistem kararsızlığı | Yerleşimde çakışma bulunmaktadır | `insane` kutu boyutu büyütülmelidir |
| `MessageError: Error: credential propagation was unsuccessful` | Drive bağlama izni verilmedi | 1. bölüm yeniden çalıştırılıp izin verilmelidir |

---

## Kaynaklar

- [Martini Protein Model — Using Martinize2](https://cgmartini.nl/docs/tutorials/Martini3/ProteinsI/Tut1.html)
- [Modeling Complex Lipid Membranes — INSANE](https://cgmartini.nl/docs/tutorials/Martini3/LipidsII/)
- [Notes and Limitations](https://cgmartini.nl/docs/tutorials/Martini3/ProteinsI/Tut4.html)
- Kuvvet alanı dosyaları: [marrink-lab/martini-forcefields](https://github.com/marrink-lab/martini-forcefields)
